In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [19]:
text="I am living in India"

In [20]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
print(tokenizer.word_index)

{'i': 1, 'am': 2, 'living': 3, 'in': 4, 'india': 5}
{'i': 1, 'am': 2, 'living': 3, 'in': 4, 'india': 5}


In [4]:
total_words= len(tokenizer.word_index)+1
print(total_words)

6


In [5]:
input_sequence = []
for i in text.split('\n'):
  token_list = tokenizer.texts_to_sequences([i])[0]
  print(token_list)

[1, 2, 3, 4, 5]


In [6]:
for i in range(1, len(token_list)):
  n_gram_sequence = token_list[:i+1]
  input_sequence.append(n_gram_sequence)

In [7]:
input_sequence

[[1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5]]

In [8]:
max_sequence_len = max([len(x) for x in input_sequence])
max_sequence_len

5

In [9]:
input_sequence= np.array(pad_sequences(input_sequence, maxlen=max_sequence_len, padding='pre'))
input_sequence

array([[0, 0, 0, 1, 2],
       [0, 0, 1, 2, 3],
       [0, 1, 2, 3, 4],
       [1, 2, 3, 4, 5]], dtype=int32)

In [10]:
predictors, target = input_sequence[:,:-1],input_sequence[:,-1]



In [11]:
predictors

array([[0, 0, 0, 1],
       [0, 0, 1, 2],
       [0, 1, 2, 3],
       [1, 2, 3, 4]], dtype=int32)

In [12]:
target

array([2, 3, 4, 5], dtype=int32)

In [13]:
target = tf.keras.utils.to_categorical(target,num_classes=total_words)
target

array([[0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 1.]])

In [14]:
from tensorflow.keras.callbacks import EarlyStopping                                 #earlystopping - used to avoid overfitting
early_stopping = EarlyStopping(monitor='loss', patience=3,restore_best_weights=True)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential()
model.add(Embedding(total_words, 100,input_length=max_sequence_len-1))                 #embedding - used to convert the words into different dimension vectors
model.add(LSTM(150,return_sequences = True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words,activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [15]:
history = model.fit(predictors, target, epochs = 50,callbacks=[early_stopping])

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step - accuracy: 0.2500 - loss: 1.7912
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.5000 - loss: 1.7814
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.5000 - loss: 1.7723
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5000 - loss: 1.7620
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.5000 - loss: 1.7498
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.5000 - loss: 1.7361
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.5000 - loss: 1.7206
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.5000 - loss: 1.7053
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.5000 - loss: 1.6812
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.5000 - loss: 1.6589
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.5000 - loss: 1.6292
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.5000 - loss: 1.5944
Epo

In [16]:
def predict_next_word(model,tokenizer,seed_text,max_sequence_len,num_predictors=3):
  token_list = tokenizer.texts_to_sequences([seed_text])[0]
  if len(token_list) > max_sequence_len -1:
    token_list = token_list[-(max_sequence_len-1):]

  padded_sequence= pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
  predictions = model.predict(padded_sequence)[0]
  top_indices =predictions.argsort()[-num_predictors:][::-1]
  word_map ={v:k for k,v in tokenizer.word_index.items()}
  predicted_words = []
  predicted_prob =[]
  for id in top_indices:
     if id in word_map:
       predicted_words.append(word_map[id])
       predicted_prob.append(predictions[id])
  return predicted_words,predicted_prob




In [17]:
seed_text="I live in"
max_sequence_len=model.input_shape[1]+1
next_words,probabilities=predict_next_word(model,tokenizer,seed_text,max_sequence_len)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step


In [18]:
print('Seed text:',seed_text)
print("Predicted Next Words :",next_words)
print("Probabilities :",probabilities)

Seed text: I live in
Predicted Next Words : ['living', 'am', 'in']
Probabilities : [np.float32(0.93556845), np.float32(0.044739235), np.float32(0.019476023)]
